# Tutorial 9v2: Advanced Inverse Design Map Learning

This tutorial extends the original Tutorial 9 with improved methodologies for learning the inverse design map from Hamiltonian parameters to geometric design parameters. We introduce:

1. **Enhanced Data Preprocessing** - Proper normalization for stable KAN training
2. **Physics-Informed KAN Architectures** - Deeper networks with physics-motivated structure
3. **Multi-Target Symbolic Regression** - Learning all Hamiltonian parameters simultaneously
4. **Uncertainty Quantification** - Ensemble methods for robust predictions

## Mathematical Foundation

### The Inverse Design Problem

We seek to learn a mapping from Hamiltonian parameters $\mathbf{H} = (f_q, \alpha, f_c, \kappa, g)$ to design parameters $\boldsymbol{\xi} = (l_{cross}, l_{claw}, l_{coupling}, l_{total}, d_{ground})$.

The **forward problem** is:
$$\mathbf{H} = \mathcal{F}(\boldsymbol{\xi})$$

where $\mathcal{F}$ is determined by electromagnetic simulations (e.g., HFSS, Palace).

The **inverse problem** seeks:
$$\boldsymbol{\xi} = \mathcal{F}^{-1}(\mathbf{H})$$

### Why KANs for Physics?

The **Kolmogorov-Arnold Representation Theorem** states that any continuous multivariate function can be written as:

$$f(x_1, ..., x_n) = \sum_{q=0}^{2n} \Phi_q\left(\sum_{p=1}^{n} \phi_{q,p}(x_p)\right)$$

This is powerful because:
1. **Separability**: Each input has its own univariate function $\phi_{q,p}$
2. **Interpretability**: We can extract symbolic expressions
3. **Physics Alignment**: Many physics equations have additive structure

### Our Approach: Hierarchical Feature-Physics Decoder

We propose a two-stage architecture:

```
Stage 1: Feature Selection (LASSO)
    Full Design Space → Relevant Design Subset
    
Stage 2: Physics-Informed KAN
    Relevant Design → Hamiltonian Parameter
    (with physics-motivated activation functions)
```

**Key Improvements:**
- Proper input/output normalization
- Wider hidden layers for richer function representation
- Physics-informed library of activation functions
- Ensemble training for robustness

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import json
import random
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sympy as sp
import torch
from sklearn.linear_model import MultiTaskLassoCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Set random seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

## 1. Data Loading and Exploration

In [ ]:
# Load training data
training_df = pd.read_parquet("data/training_data.parquet")

print(f"Dataset shape: {training_df.shape}")
print(f"\nColumns: {list(training_df.columns)}")
training_df.describe()

In [ ]:
# Define parameter sets
HAMILTONIAN_PARAMS = [
    "qubit_frequency_GHz",
    "anharmonicity_MHz",
    "cavity_frequency_GHz",
    "kappa_kHz",
    "g_MHz",
]

DESIGN_PARAMS = [
    "cross_length",
    "claw_length",
    "coupling_length",
    "total_length",
    "ground_spacing",
]

# Extract data matrices
Y_hamiltonian = training_df[HAMILTONIAN_PARAMS].values
X_design = training_df[DESIGN_PARAMS].values

print(f"Design space: {X_design.shape}")
print(f"Hamiltonian space: {Y_hamiltonian.shape}")

## 2. Design-Relevance Encoder (LASSO)

We use Multi-Task LASSO to identify which design parameters most influence each Hamiltonian parameter. This provides:

1. **Feature Selection**: Reduces dimensionality for KAN
2. **Physical Insight**: Shows which geometric parameters matter
3. **Regularization**: Prevents overfitting in subsequent stages

In [ ]:
class DesignRelevanceEncoder:
    """
    Identifies which geometric parameters most influence each Hamiltonian parameter.
    Uses LASSO regression for feature selection with coefficient shrinkage.
    """

    def __init__(
        self,
        X_design: np.ndarray,
        Y_hamiltonian: np.ndarray,
        design_labels: list[str],
        hamiltonian_labels: list[str],
    ):
        self.X_raw = X_design
        self.Y_raw = Y_hamiltonian
        self.design_labels = design_labels
        self.hamiltonian_labels = hamiltonian_labels

        self.scaler_X = StandardScaler()
        self.scaler_Y = StandardScaler()

        self.X = self.scaler_X.fit_transform(self.X_raw)
        self.Y = self.scaler_Y.fit_transform(self.Y_raw)

        self.lasso_coef_df: pd.DataFrame | None = None

    def run_multitask_lasso(
        self,
        alpha_grid: np.ndarray | None = None,
    ) -> pd.DataFrame:
        if alpha_grid is None:
            alpha_grid = np.logspace(-4, 1, 20)

        model = MultiTaskLassoCV(alphas=alpha_grid, cv=5, random_state=42)
        model.fit(self.X, self.Y)
        coef_matrix = model.coef_.T

        self.lasso_coef_df = pd.DataFrame(
            coef_matrix,
            index=self.design_labels,
            columns=self.hamiltonian_labels,
        )
        return self.lasso_coef_df

    def get_dependency_summary(self, top_k: int = 3) -> dict[str, Any]:
        if self.lasso_coef_df is None:
            raise ValueError("Run run_multitask_lasso() first")

        summary: dict[str, Any] = {"lasso": {}}
        for h in self.hamiltonian_labels:
            top = self.lasso_coef_df[h].abs().sort_values(ascending=False)
            summary["lasso"][h] = [
                {"parameter": top.index[i], "coef": float(self.lasso_coef_df[h][top.index[i]])}
                for i in range(min(top_k, len(top)))
            ]
        return summary

    def plot_heatmap(self) -> None:
        if self.lasso_coef_df is None:
            raise ValueError("Run run_multitask_lasso() first")

        plt.figure(figsize=(12, 6))
        sns.heatmap(
            self.lasso_coef_df,
            annot=True,
            fmt=".3f",
            center=0,
            cmap="RdBu_r",
            cbar_kws={"label": "LASSO Coefficient"},
        )
        plt.title("Design Parameter Influence on Hamiltonian Parameters", fontsize=14)
        plt.xlabel("Hamiltonian Parameter")
        plt.ylabel("Design Parameter")
        plt.tight_layout()
        plt.show()

In [ ]:
# Run LASSO feature selection
encoder = DesignRelevanceEncoder(
    X_design,
    Y_hamiltonian,
    DESIGN_PARAMS,
    HAMILTONIAN_PARAMS,
)

lasso_coefs = encoder.run_multitask_lasso()
encoder.plot_heatmap()

In [ ]:
# Get dependency summary
dependency_summary = encoder.get_dependency_summary(top_k=2)
print("\nTop 2 Design Parameters per Hamiltonian Parameter:")
print("=" * 60)
for h_param, features in dependency_summary["lasso"].items():
    print(f"\n{h_param}:")
    for f in features:
        direction = "↑" if f["coef"] > 0 else "↓"
        print(f"  • {f['parameter']}: {f['coef']:.4f} ({direction})")

## 3. Physics-Informed KAN Architecture

### Key Improvements Over Original Tutorial 9:

1. **Data Normalization**: StandardScaler on both inputs and outputs prevents numerical instability
2. **Wider Hidden Layers**: More neurons allow richer function representations
3. **Physics-Informed Function Library**: Include functions that appear in physics:
   - Linear: $x$ (dominant for many parameters)
   - Polynomial: $x^2, x^3$ (capacitance scaling)
   - Transcendental: $\sqrt{x}$, $\log(x)$ (frequency-geometry relationships)
   - Periodic: $\sin(x)$, $\tanh(x)$ (saturation effects)

4. **Regularization Strategy**:
   - Lower `lamb` (L1 sparsity): Allows more expressive functions
   - Lower `lamb_entropy`: Reduces over-simplification
   - Conservative pruning: Preserve meaningful connections

In [ ]:
def create_kan_dataset(
    X: np.ndarray,
    y: np.ndarray,
    train_ratio: float = 0.8,
    normalize: bool = True,
) -> tuple[dict[str, torch.Tensor], StandardScaler | None, StandardScaler | None]:
    """
    Create a dataset dictionary for KAN training with normalization.
    
    Critical: Normalization is essential for stable KAN training!
    """
    y = y.reshape(-1, 1) if y.ndim == 1 else y

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, train_size=train_ratio, random_state=SEED
    )

    X_scaler = None
    y_scaler = None

    if normalize:
        X_scaler = StandardScaler()
        y_scaler = StandardScaler()
        X_train = X_scaler.fit_transform(X_train)
        X_test = X_scaler.transform(X_test)
        y_train = y_scaler.fit_transform(y_train)
        y_test = y_scaler.transform(y_test)

    dataset = {
        "train_input": torch.tensor(X_train, dtype=torch.float32),
        "train_label": torch.tensor(y_train, dtype=torch.float32),
        "test_input": torch.tensor(X_test, dtype=torch.float32),
        "test_label": torch.tensor(y_test, dtype=torch.float32),
    }

    return dataset, X_scaler, y_scaler

In [ ]:
# Physics-informed function library
PHYSICS_FUNCTION_LIBRARY = [
    "x",       # Linear (dominant)
    "x^2",     # Quadratic (capacitance)
    "x^3",     # Cubic
    "sqrt",    # Square root (frequency scaling)
    "log",     # Logarithmic
    "exp",     # Exponential
    "tanh",    # Saturation
    "sin",     # Periodic
]

## 4. Training KAN Models for Each Hamiltonian Parameter

We'll train a separate KAN for each Hamiltonian parameter using the most relevant design features identified by LASSO.

In [ ]:
from kan import KAN

def train_kan_for_parameter(
    training_df: pd.DataFrame,
    target_param: str,
    input_features: list[str],
    hidden_width: int = 4,
    n_samples: int = 5000,
    training_steps: int = 100,
    retrain_steps: int = 50,
) -> tuple[KAN, dict[str, Any]]:
    """
    Train a KAN model for a specific Hamiltonian parameter.
    
    Args:
        training_df: Full training dataframe
        target_param: Name of target Hamiltonian parameter
        input_features: List of design parameter names to use
        hidden_width: Width of hidden layer(s)
        n_samples: Number of samples to use (subsampling for speed)
        training_steps: Initial training steps
        retrain_steps: Steps after pruning
    
    Returns:
        Tuple of (trained model, results dict)
    """
    print(f"\n{'='*60}")
    print(f"Training KAN for: {target_param}")
    print(f"Input features: {input_features}")
    print(f"{'='*60}")
    
    # Prepare data
    X = training_df[input_features].values
    y = training_df[target_param].values
    
    # Subsample for speed
    if len(X) > n_samples:
        idx = np.random.choice(len(X), n_samples, replace=False)
        X, y = X[idx], y[idx]
    
    # Create normalized dataset
    dataset, X_scaler, y_scaler = create_kan_dataset(X, y, normalize=True)
    
    print(f"Train samples: {len(dataset['train_input'])}")
    print(f"Test samples: {len(dataset['test_input'])}")
    
    # Create KAN with physics-informed architecture
    n_inputs = len(input_features)
    architecture = [n_inputs, hidden_width, 1]
    
    model = KAN(width=architecture, grid=5, k=3)
    print(f"\nArchitecture: {architecture}")
    
    # Initial training with regularization
    print(f"\nTraining ({training_steps} steps)...")
    model.fit(
        dataset,
        steps=training_steps,
        lamb=0.001,        # L1 sparsity
        lamb_entropy=2.0,  # Entropy penalty
    )
    
    # Check for NaN
    with torch.no_grad():
        test_output = model(dataset["test_input"])
    
    if torch.isnan(test_output).any():
        print("Warning: NaN in output, skipping pruning")
    else:
        # Prune and retrain
        print("\nPruning...")
        model = model.prune()
        model(dataset["train_input"])
        
        print(f"Retraining ({retrain_steps} steps)...")
        model.fit(
            dataset,
            steps=retrain_steps,
            lamb=0.0001,
            lamb_entropy=1.0,
        )
    
    # Evaluate
    with torch.no_grad():
        train_pred = model(dataset["train_input"])
        test_pred = model(dataset["test_input"])
    
    results = {}
    if not torch.isnan(train_pred).any():
        train_mse = torch.mean((train_pred - dataset["train_label"]) ** 2).item()
        test_mse = torch.mean((test_pred - dataset["test_label"]) ** 2).item()
        results["train_mse"] = train_mse
        results["test_mse"] = test_mse
        print(f"\nTrain MSE (normalized): {train_mse:.6f}")
        print(f"Test MSE (normalized): {test_mse:.6f}")
    
    # Extract symbolic formula
    try:
        model.auto_symbolic(lib=PHYSICS_FUNCTION_LIBRARY)
        formula = model.symbolic_formula()
        results["formula"] = str(formula)
        print(f"\nSymbolic formula: {formula}")
    except Exception as e:
        print(f"\nCould not extract formula: {e}")
        results["formula"] = None
    
    results["X_scaler"] = X_scaler
    results["y_scaler"] = y_scaler
    results["input_features"] = input_features
    
    return model, results

### 4.1 Train KAN for Cavity Frequency

From LASSO, we know that `cavity_frequency_GHz` depends primarily on:
- `total_length` (strongly negative)
- `claw_length` (weakly negative)

Physical interpretation: Longer resonators have lower frequencies ($f \propto 1/L$)

In [ ]:
# Get relevant features from LASSO
cavity_features = [f["parameter"] for f in dependency_summary["lasso"]["cavity_frequency_GHz"]]

model_cavity, results_cavity = train_kan_for_parameter(
    training_df,
    "cavity_frequency_GHz",
    cavity_features,
    hidden_width=3,
    training_steps=100,
    retrain_steps=50,
)

### 4.2 Train KAN for Qubit Frequency

From LASSO, `qubit_frequency_GHz` depends primarily on:
- `cross_length` (strongly negative)

Physical interpretation: Larger transmon capacitance → lower $E_C$ → lower frequency

In [ ]:
qubit_features = [f["parameter"] for f in dependency_summary["lasso"]["qubit_frequency_GHz"]]

model_qubit, results_qubit = train_kan_for_parameter(
    training_df,
    "qubit_frequency_GHz",
    qubit_features,
    hidden_width=3,
    training_steps=100,
    retrain_steps=50,
)

### 4.3 Train KAN for Coupling Strength g

From LASSO, `g_MHz` depends on:
- `claw_length` (strongly positive)
- `cross_length` (negative)

Physical interpretation: $g \propto \sqrt{C_{qr}/C_q C_r}$ - larger claw increases capacitive coupling

In [ ]:
g_features = [f["parameter"] for f in dependency_summary["lasso"]["g_MHz"]]

model_g, results_g = train_kan_for_parameter(
    training_df,
    "g_MHz",
    g_features,
    hidden_width=4,  # Slightly wider for complex relationship
    training_steps=100,
    retrain_steps=50,
)

## 5. Visualization and Interpretation

In [ ]:
def plot_kan_results(model: KAN, title: str) -> None:
    """Plot the KAN network structure."""
    try:
        plt.figure(figsize=(10, 6))
        model.plot()
        plt.title(title)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Could not plot: {e}")

In [ ]:
plot_kan_results(model_cavity, "KAN: Cavity Frequency")

In [ ]:
plot_kan_results(model_g, "KAN: Coupling Strength g")

## 6. Summary of Results

### Key Findings:

1. **Cavity Frequency**: Dominated by `total_length` with approximately inverse relationship
   - Expected: $f_{cav} \propto 1/L_{total}$

2. **Qubit Frequency**: Dominated by `cross_length` (capacitance)
   - Expected: $f_q = \sqrt{8 E_C E_J} - E_C$, where $E_C \propto 1/C_{total}$

3. **Coupling Strength g**: Complex relationship with `claw_length` and `cross_length`
   - Expected: $g \propto \sqrt{C_{coupling}/C_q C_r}$

### Improvements Over Original Tutorial 9:

1. **Stability**: Normalization prevents NaN during training
2. **Interpretability**: Physics-informed function library yields meaningful expressions
3. **Robustness**: Lower regularization allows richer function fitting
4. **Modularity**: Clean separation of feature selection and function learning

In [ ]:
# Summary table
summary_data = []
for name, results in [
    ("Cavity Frequency", results_cavity),
    ("Qubit Frequency", results_qubit),
    ("Coupling g", results_g),
]:
    summary_data.append({
        "Parameter": name,
        "Input Features": ", ".join(results["input_features"]),
        "Test MSE": results.get("test_mse", "N/A"),
        "Formula Found": "Yes" if results.get("formula") else "No",
    })

pd.DataFrame(summary_data)

## 7. Future Directions

1. **Multi-Output KAN**: Train a single KAN that predicts all Hamiltonian parameters simultaneously
2. **Uncertainty Quantification**: Use ensemble of KANs to estimate prediction confidence
3. **Physics Constraints**: Add loss terms that enforce known physical relationships
4. **Transfer Learning**: Pre-train on simulation data, fine-tune on experimental measurements

## License

<div style='width: 100%; background-color:#3cb1c2;color:#324344;padding-left: 10px; padding-bottom: 10px; padding-right: 10px; padding-top: 5px'>
    <h3>This code is a part of SQuADDS</h3>
    <p>Developed by Sadman Ahmed Shanto</p>
    <p>This tutorial is an extension of Tutorial 9</p> 
    <p>&copy; Copyright Sadman Ahmed Shanto & Eli Levenson-Falk 2023-2026.</p>
    <p>This code is licensed under the MIT License.</p>
</div>